## Train/Test Split

In this part we split the cleaned dataset into a train set and a test set, **before** any EDA, so that the EDA and every choice made from it (PCA, kept columns, target definition thresholds) only see the train set.

### Changes

- **Rows without any target:** The 105 rows with no tensile result and no Charpy result are dropped, they cannot be used for supervised learning.
- **weld_group:** New column, one id per weld deposit (same chemistry and same welding parameters). It is the group used for the split and later for `GroupKFold`.
- **Split:** 80% train / 20% test, by `weld_group`, stratified on which targets are available.

### Operations in order

1. Load `../data/welddb_cleaned_units.csv`, the output of the units notebook.
2. Show that one weld is spread over several rows with different `weld_id`.
3. Drop the rows without any target.
4. Build `weld_group` from the chemistry and the welding parameters.
5. Split with `StratifiedGroupKFold`, the first fold is the test set.
6. Check that no group is in both sets and that the targets are available in the same proportions.
7. Save to `../data/train.csv` and `../data/test.csv`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

In [3]:
df = pd.read_csv("../data/welddb_cleaned_units.csv")

tensile_cols = ["yield_strength_MPa", "uts_MPa", "elongation_pct", "reduction_area_pct"]
target_cols = tensile_cols + ["charpy_toughness_J"]

print(df.shape)

(1639, 37)


`charpy_temp_C` is not in `target_cols`: it is the temperature at which the Charpy test was run, so it is a test condition, not a result.

In [4]:
df.loc[df["weld_id"].str.startswith("Evans-Ni/CMn-1990/1991-0A"), ["weld_id", "pwht_temp_C"] + target_cols + ["charpy_temp_C"]]

,weld_id,pwht_temp_C,yield_strength_MPa,uts_MPa,elongation_pct,reduction_area_pct,charpy_toughness_J,charpy_temp_C
0,Evans-Ni/CMn-1990/1991-0Aaw,250.0,392.0,466.0,31.9,80.6,NaN,NaN
1,Evans-Ni/CMn-1990/1991-0Aawch,0.0,NaN,NaN,NaN,NaN,100.0,-28.0
2,Evans-Ni/CMn-1990/1991-0Aht,580.0,370.0,456.0,35.2,80.6,100.0,-38.0


- **One weld, several `weld_id`:** `0Aaw`, `0Aawch` and `0Aht` are the same weld deposit (same chemistry, same welding parameters). The tensile result is on `0Aaw`, the Charpy result of the as-welded state on `0Aawch` (`ch` for Charpy), and `0Aht` is the same deposit after a 580°C heat treatment. The 250°C of `0Aaw` is most likely the hydrogen removal ageing of the tensile specimens, not a real heat treatment.

Splitting on `weld_id` would therefore put `0Aaw` in train and `0Aawch` in test: the model would have seen the exact composition of the test weld. A random split on the rows has the same problem.

In [5]:
no_target = df[target_cols].isna().all(axis=1)
print(no_target.sum())

df = df[~no_target].reset_index(drop=True)
print(df.shape)

105
(1534, 37)


- **Rows without any target:** Dropped, they have neither a tensile result nor a Charpy result (their hardness or FATT50 columns were dropped in the missingness notebook).

In [6]:
chem_cols = ["C", "Si", "Mn", "S", "P", "Ni", "Cr", "Mo", "V", "O_ppm", "Ti_ppm", "N_ppm", "Al_ppm", "Nb_ppm"]
process_cols = ["current_A", "voltage_V", "AC_DC", "electrode_polarity", "AC_DC_unknown", "heat_input_kJ_mm", "interpass_temp_C"]
process_cols += [c for c in df.columns if c.startswith("weld_type_")]

df["weld_group"] = df.groupby(chem_cols + process_cols).ngroup()

print(df["weld_id"].nunique(), df["weld_group"].nunique())
df["weld_group"].value_counts().value_counts().sort_index()

1383 602


count
1     226
2     110
3     176
4      10
5      20
6      35
7      16
8       2
9       1
10      3
11      1
12      1
20      1
Name: count, dtype: int64

- **weld_group:** Two rows belong to the same group when they have the same chemistry and the same welding parameters. `pwht_temp_C`, `pwht_time_h` and `charpy_temp_C` are left out on purpose, so that the as-welded row, the heat treated row and every Charpy row of one deposit end up in the same group.

The second table gives the number of groups per group size: most deposits have 1 to 3 rows, a few have up to 20 (a Charpy curve tested at many temperatures).

Two different welds whose missing values were imputed with the same median can land in the same group. This only makes some groups bigger, which is the safe direction.

A stricter option is to group by paper, to test if the model generalises to a new study. The paper is not cleanly written in `weld_id` (`Evans` covers many papers), so we keep the deposit grouping.

In [7]:
has_tensile = df[tensile_cols].notna().any(axis=1)
has_charpy = df["charpy_toughness_J"].notna()
target_profile = has_tensile.astype(int) + 2 * has_charpy.astype(int)

print(target_profile.value_counts())

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, target_profile, groups=df["weld_group"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print(len(train), len(test), round(len(test) / len(df), 3))

2    735
1    655
3    144
Name: count, dtype: int64
1229 305 0.199


- **target_profile:** 1 = tensile only, 2 = Charpy only, 3 = both. Stratifying on it keeps the same share of tensile and Charpy rows in the two sets, whatever target we choose later.

- **StratifiedGroupKFold:** 5 folds, so the first fold holds about 20% of the groups. It is used once here, only to get a split that respects both the groups and the stratification.

In [8]:
assert set(train["weld_group"]).isdisjoint(test["weld_group"])

display(pd.DataFrame({"train": train[target_cols].notna().mean(), "test": test[target_cols].notna().mean()}).round(2))
display(pd.DataFrame({"train": train[target_cols].mean(), "test": test[target_cols].mean()}).round(1))

,train,test
yield_strength_MPa,0.50,0.51
uts_MPa,0.48,0.48
elongation_pct,0.45,0.45
reduction_area_pct,0.45,0.46
charpy_toughness_J,0.57,0.57


,train,test
yield_strength_MPa,505.3,510.5
uts_MPa,592.3,602.7
elongation_pct,26.4,25.9
reduction_area_pct,72.1,71.7
charpy_toughness_J,84.3,101.4


No group is in both sets, and each target is available in the same proportion in train and test. The target means are close, `charpy_toughness_J` varies the most because the Charpy results come in large groups.

In [9]:
train.to_csv("../data/train.csv", index=False)
test.to_csv("../data/test.csv", index=False)

print(train.shape, test.shape)

(1229, 38) (305, 38)


- **weld_group:** Kept in both files. The cross-validation on the train set must use it (`GroupKFold(groups=train["weld_group"])`), for the same reason as this split.

- **Imputation:** The medians of the missingness notebook (`S`, `P`, `O_ppm`, `N_ppm`, `Al_ppm`, `current_A`, `voltage_V`) were computed on the whole dataset. It only concerns inputs, not targets, so the leakage is small, but to be strict the median imputation should be moved into a `Pipeline` fitted on the train set only.

- **From now on the test set is not opened** until the final evaluation: EDA, PCA, scaling and imputation are fitted on `train.csv` only.